In [2]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

# ==========================================
# 1. GENERATE MOCK DATA
# ==========================================
# Creating an imbalanced dataset: 85% Class 0, 15% Class 1
X, y = make_classification(n_samples=1000, n_features=20, weights=[0.85, 0.15], random_state=42)

# ==========================================
# 2. DEFINE THE ARCHITECTURE (THE PIPELINE)
# ==========================================
# Locking preprocessing and modeling together prevents data leakage during CV
pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(max_depth=10, random_state=42))
])

# ==========================================
# 3. DEFINE THE SPLITTING STRATEGY
# ==========================================
# StratifiedKFold guarantees the 85/15 ratio is preserved in every single fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ==========================================
# 4. EXECUTE ADVANCED CROSS-VALIDATION
# ==========================================
print("Running 5-Fold Cross-Validation...")

# cross_validate returns a dictionary containing arrays of times and scores
cv_results = cross_validate(
    estimator=pipeline,
    X=X,
    y=y,
    cv=skf,
    scoring=['accuracy', 'precision', 'recall', 'f1'], # Calculate 4 different metrics
    return_train_score=True,                           # Critical for checking if the model memorized the data
    n_jobs=-1                                          # -1 uses all available CPU cores for parallel training
)

# ==========================================
# 5. DIAGNOSE THE RESULTS
# ==========================================
print("\n--- Production Metrics (Average across 5 Folds) ---")
print(f"Mean Test Accuracy : {cv_results['test_accuracy'].mean():.4f}")
print(f"Mean Test Precision: {cv_results['test_precision'].mean():.4f}")
print(f"Mean Test Recall   : {cv_results['test_recall'].mean():.4f}")
print(f"Mean Test F1-Score : {cv_results['test_f1'].mean():.4f}")

print("\n--- Overfitting Diagnostics ---")
train_f1 = cv_results['train_f1'].mean()
test_f1 = cv_results['test_f1'].mean()
print(f"Average Train F1   : {train_f1:.4f}")
print(f"Average Test F1    : {test_f1:.4f}")
print(f"Overfitting Gap    : {(train_f1 - test_f1):.4f} (Closer to 0 is better)")
print("HEllllo")

Running 5-Fold Cross-Validation...

--- Production Metrics (Average across 5 Folds) ---
Mean Test Accuracy : 0.9450
Mean Test Precision: 0.8867
Mean Test Recall   : 0.7301
Mean Test F1-Score : 0.7987

--- Overfitting Diagnostics ---
Average Train F1   : 0.9892
Average Test F1    : 0.7987
Overfitting Gap    : 0.1905 (Closer to 0 is better)
HEllllo
